# 16.2 Peer-Review — 'test 정확도 95%'는 좋은 딥러닝 모델인가, 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter16_2_deep_review.ipynb)

책 본문: [16.2 Peer-Review](https://smhanlab.com/book-ml/kor/ml1/chapter16/2.html)

본문이 놓은 '가상의 딥러닝 발표'("breast_cancer(569건)에 MLP를 써서 test 정확도 0.95,
로지스틱회귀(0.94)·GBDT(0.94)보다 높다")를 코드로 재현합니다. (1) 세 모델의 정확도 차이가
**Ch06.3의 선택 편향 잡음과 같은 크기**인지, (2) '에폭 50'을 정한 **학습 곡선**이 실제로
어떤 모양인지, (3) 정확도 0.95의 모델이 **교정(calibration)**이 되어 있는가, (4) 이 숫자들로
**반박 가능한 리뷰 질문**을 어떻게 짜는지 순서대로 확인합니다. numpy/torch/cpu, scikit-learn,
matplotlib만 쓰며 모든 시드가 고정되어 있습니다. (Colab에서는 첫 셀의 `IMG` 경로를 `/tmp`로
바꾸면 됩니다.)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

## 1. 데이터: breast_cancer (569 × 30, 정형 표 데이터)

발표의 설정을 그대로 씁니다 — breast_cancer(악성 63%:양성 37%)에 *정형 표 데이터*를
다루는 MLP(완전연결 1층). train/test를 70/30으로 stratify 분할하고, 전처리(`StandardScaler`)의
`fit`은 **train으로만** 합니다(Ch06.3의 원칙 — 8.1절의 "자주 하는 실수" 항목).

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)                 # <- fit은 train으로만 (Ch06.3)
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

print(f"train {len(Xtr)}건, test {len(Xte)}건")
print(f"클래스 비율: 악성 {y.mean():.3f} / 양성 {1-y.mean():.3f}")
print(f"테스트셋 다수 클래스(악성) 비율 = {yte.mean():.3f}  <- '무조건 악성' 베이스라인 정확도")


train 398건, test 171건
클래스 비율: 악성 0.627 / 양성 0.373
테스트셋 다수 클래스(악성) 비율 = 0.626  <- '무조건 악성' 베이스라인 정확도


## 2. 같은 분할에서 세 모델 — '정확도 0.95 vs 0.94'는 얼마나 큰 차이인가

발표는 "MLP 0.95 > 로지스틱회귀 0.94 = GBDT 0.94이므로 MLP가 더 좋다"고 결론 냈습니다.
같은 train/test 분할에 세 모델을 돌려 정확도·F1·PR-AUC·AUC를 비교합니다 —
지표가 하나(정확도)만 아니라 **여러 개**일 때 순위가 어떻게 달라지는지 보는 것이지,
"어떤 모델이 진짜로 더 좋은가"를 가리는 실험이 **아닙니다**(그걸 가리려면
Ch06.2의 교차검증이 필요 — §4에서 그 이유를 숫자로 확인).

MLP: 30 → 64(ReLU, dropout 0.5) → 2, Adam(1e-3), 배치 64, 50에폭 — 발표가 말한 설정 그대로.
교정 확인을 위해 최종 예측 확률은 **dropout off** 상태로 뽑습니다.

In [3]:
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, roc_auc_score, log_loss

torch.manual_seed(0)
Xt  = torch.tensor(Xtr, dtype=torch.float32); yt  = torch.tensor(ytr, dtype=torch.long)
Xtt = torch.tensor(Xte, dtype=torch.float32); ytt = torch.tensor(yte, dtype=torch.long)

mlp = nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 2))
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

for epoch in range(50):
    mlp.train()
    idx = torch.randperm(len(yt))
    for i in range(0, len(yt), 64):
        b = idx[i:i+64]
        opt.zero_grad()
        loss = lossfn(mlp(Xt[b]), yt[b])
        loss.backward()
        opt.step()

mlp.eval()
with torch.no_grad():
    prob_mlp = torch.softmax(mlp(Xtt), 1)[:, 1].numpy()      # dropout off

lr = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=0).fit(Xtr, ytr)

rows = [
    ("다수 클래스 (악성만 찍음)", np.ones(len(yte)), yte),   # 베이스라인: 무조건 악성 (Ch08.1)
    ("로지스틱회귀 (Ch02)", lr.predict_proba(Xte)[:, 1], yte),
    ("GBDT (Ch07.3)",        gb.predict_proba(Xte)[:, 1], yte),
    ("MLP 30-64-2 (Ch09)",   prob_mlp, yte),
]
print(f"{'모델':22s} {'accuracy':>9} {'F1':>7} {'PR-AUC':>7} {'AUC':>7} {'log-loss@0.5':>13}")
for name, prob, yy in rows:
    pred = (prob > 0.5).astype(int)
    ll = f"{log_loss(yy, pred, labels=[0,1]):13.4f}" if 0 in pred and 1 in pred else f"{'(미정의)':>13s}"
    print(f"{name:22s} {accuracy_score(yy, pred):9.4f} {f1_score(yy, pred):7.4f} "
          f"{average_precision_score(yy, prob):7.4f} {roc_auc_score(yy, prob):7.4f} "
          f"{ll}")
print()
mlp_params = 30*64 + 64 + 64*2 + 2
gb_nodes = sum(len(t.tree_.feature) for t in gb.estimators_.ravel())
print(f"파라미터 수: 로지스틱회귀=31, GBDT(200 트리, depth 3) 노드 수≈{gb_nodes}, MLP={mlp_params}")
accs3 = [accuracy_score(yte, (p > 0.5)) for _, p, _ in rows[1:]]
print(f"세 모델(로지스틱·GBDT·MLP) 간 정확도 차이(max-min) = {max(accs3) - min(accs3):.4f}")


모델                      accuracy      F1  PR-AUC     AUC  log-loss@0.5
다수 클래스 (악성만 찍음)           0.6257  0.7698  0.6257  0.5000         (미정의)
로지스틱회귀 (Ch02)             0.9591  0.9671  0.9974  0.9956        1.4755
GBDT (Ch07.3)             0.9415  0.9528  0.9922  0.9880        2.1078
MLP 30-64-2 (Ch09)        0.9532  0.9626  0.9958  0.9930        1.6863

파라미터 수: 로지스틱회귀=31, GBDT(200 트리, depth 3) 노드 수≈2874, MLP=2114
세 모델(로지스틱·GBDT·MLP) 간 정확도 차이(max-min) = 0.0175


## 3. '에폭 50'이 val 곡선의 포화 지점인가 — 학습 곡선

발표의 "에폭 50"이 Ch09.3의 early stopping 기준(val 곡선의 정점)으로 정해진 숫자인지
확인하려면 학습 곡선이 필요합니다(16.1의 M3 마일스톤이 이를 요구하는 이유).
train을 279/119로 다시 나눠 **val 곡선**을 그립니다 — val 곡선이 언제 포화(saturate)
되는지, 그 이후에 train 손실만 계속 떨어지면서 train/val 손실 간격이 벌어지는지
(Ch08.1 M3 — 에폭도 하이퍼파라미터이므로 val 곡선으로만 정한다. Ch09.3의
과적합 신호와도 같은 모양)를 봅니다.

**리뷰어가 여기서 해야 할 일**: val 곡선이 포화되는 에폭을 읽고, "발표가 쓴
50에폭이 그 에폭인가?"를 묻는 것입니다.

In [4]:
Xc_tr, Xc_va, yc_tr, yc_va = train_test_split(Xtr, ytr, test_size=119, random_state=1)
Xtc = torch.tensor(Xc_tr, dtype=torch.float32); ytc = torch.tensor(yc_tr, dtype=torch.long)
Xvc = torch.tensor(Xc_va, dtype=torch.float32); yvc = torch.tensor(yc_va, dtype=torch.long)

mlp2 = nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 2))
opt2 = torch.optim.Adam(mlp2.parameters(), lr=1e-3)

tr_acc, va_acc, tr_ll, va_ll = [], [], [], []
for epoch in range(50):
    mlp2.train()
    for i in range(0, len(ytc), 64):
        b = torch.randperm(len(ytc))[i:i+64]
        opt2.zero_grad()
        loss = lossfn(mlp2(Xtc[b]), ytc[b])
        loss.backward()
        opt2.step()
    mlp2.eval()
    with torch.no_grad():
        ot, ov = mlp2(Xtc), mlp2(Xvc)
        tr_acc.append(float((ot.argmax(1) == ytc).float().mean()))
        va_acc.append(float((ov.argmax(1) == yvc).float().mean()))
        tr_ll.append(float(lossfn(ot, ytc).item()))
        va_ll.append(float(lossfn(ov, yvc).item()))

best = max(va_acc)
sat_epoch = int(np.argmax(np.array(va_acc) >= best - 1e-9)) + 1
print(f"val 정확도 최고치: {best:.4f}  (에폭 {sat_epoch}부터 도달 — val 곡선 포화)")
print(f"에폭 50에서 val acc = {va_acc[-1]:.4f}")
print(f"train 손실은 계속 떨어짐 ({tr_ll[0]:.3f} -> {tr_ll[-1]:.3f}), val 손실은 {min(va_ll):.3f} 부근에서 멈춤 (train-val 간격이 벌어짐)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
epochs = np.arange(1, 51)
ax = axes[0]
ax.plot(epochs, tr_acc, color="#1d4ed8", lw=2, label="train accuracy")
ax.plot(epochs, va_acc, color="#dc2626", lw=2, label="val accuracy")
ax.axvline(sat_epoch, color="#0f5132", ls=":", lw=1.2)
ax.text(sat_epoch + 1, 0.66, f"val saturates: epoch {sat_epoch} ({best:.3f})", fontsize=9, color="#0f5132")
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
ax.set_title("Accuracy: val saturates while train keeps rising")
ax.grid(alpha=0.3); ax.legend(fontsize=9)
ax = axes[1]
ax.plot(epochs, tr_ll, color="#1d4ed8", lw=2, label="train cross-entropy")
ax.plot(epochs, va_ll, color="#dc2626", lw=2, label="val cross-entropy")
ax.axvline(sat_epoch, color="#0f5132", ls=":", lw=1.2)
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy (Ch09.2)")
ax.set_title("Loss: val plateaus at the floor while train keeps decreasing (Ch09.3 overfitting)")
ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch16_2_mlp_epoch_curve.svg")
plt.show()


val 정확도 최고치: 1.0000  (에폭 32부터 도달 — val 곡선 포화)
에폭 50에서 val acc = 0.9916
train 손실은 계속 떨어짐 (0.702 -> 0.050), val 손실은 0.052 부근에서 멈춤 (train-val 간격이 벌어짐)


## 4. '0.95 vs 0.94'는 잡음인가 — 같은 모델, 같은 가중치, test 표본만 바꿔보기

Ch06.3의 논리를 가장 직접적으로 보여주는 실험입니다. **같은 로지스틱회귀 모델(같은
가중치)**을 두고, test 풀(171건)에서 100건을 무작위로 뽑아 정확도를 60번 계산합니다.
모델이 전혀 바뀌지 않았는데 정확도가 어느 범위에서 흔들리는지 — 그 범위가
§2의 "세 모델 간 정확도 차이"보다 크다면, 그 차이는 *모델이 아니라 표본*에서
나온 것입니다.

In [5]:
rng = np.random.RandomState(0)
proba_te = lr.predict_proba(Xte)[:, 1]
pool = np.arange(len(yte))

accs = []
for k in range(60):
    sub = rng.choice(pool, size=100, replace=False)
    accs.append(accuracy_score(yte[sub], (proba_te[sub] > 0.5)))
accs = np.array(accs)
print("같은 모델(같은 가중치), test 100건 하위샘플 60회:")
print(f"  정확도 범위: {accs.min():.4f} ~ {accs.max():.4f}  (범위 {accs.max()-accs.min():.4f})")
print(f"  표준편차: {accs.std():.4f}")
print()
d_lr  = accuracy_score(yte, (proba_te > 0.5))
d_mlp = accuracy_score(yte, (prob_mlp > 0.5))
d_gb  = accuracy_score(yte, (gb.predict_proba(Xte)[:, 1] > 0.5))
gap = max(d_lr, d_mlp, d_gb) - min(d_lr, d_mlp, d_gb)
print(f"§2의 세 모델 간 정확도 차이(max-min) = {gap:.4f}")
print(f"표본 흔들림(위 범위 {accs.max()-accs.min():.4f}) vs 모델 차이({gap:.4f}):",
      "모델 차이가 표본 흔들림보다 작거나 같음 -> '정확도 1위'는 표본의 선택에 의존"
      if (accs.max() - accs.min()) >= gap else "모델 차이가 더 큼")


같은 모델(같은 가중치), test 100건 하위샘플 60회:
  정확도 범위: 0.9300 ~ 0.9900  (범위 0.0600)
  표준편차: 0.0126

§2의 세 모델 간 정확도 차이(max-min) = 0.0175
표본 흔들림(위 범위 0.0600) vs 모델 차이(0.0175): 모델 차이가 표본 흔들림보다 작거나 같음 -> '정확도 1위'는 표본의 선택에 의존


## 5. 교정: '0.95의 정확도'와 '0.99 확률의 신뢰도'는 별개의 숫자

정확도(분류)와 교정(확률의 신뢰도)이 항상 같은 것은 아닙니다. MLP가 예측 확률을
크게 묶어(0.50~0.60, 0.60~0.80, 0.80~0.90, 0.90~1.00) 각각 **실제**로 악성이
맞은 비율을 세어봅니다 — "0.99 확률로 예측한 것"이 실제로 99% 맞아야 교정이
된 것입니다(Ch09.2의 교차엔트로피가 낮아도 확률 스케일 전체는 어긋날 수 있음).

In [6]:
bins = [(0.50, 0.60), (0.60, 0.80), (0.80, 0.90), (0.90, 1.0001)]
print(f"{'예측 확률 구간':16s} {'샘플 수':>8} {'실제 악성 비율':>14}")
for lo, hi in bins:
    m = (prob_mlp >= lo) & (prob_mlp < hi)
    if m.sum() > 0:
        print(f"[{lo:.2f}, {hi:.2f})       {m.sum():8d} {yte[m].mean():14.4f}")
print()
top = prob_mlp >= 0.9
print(f"'0.9 이상 확률'로 예측한 {top.sum()}건 중 실제로 악성이 {yte[top].mean():.4f} — "
      f"이 숫자가 0.9에 가까운가(교정)가, 정확도 {accuracy_score(yte,(prob_mlp>0.5)):.4f}와는 별개의 질문입니다.")


예측 확률 구간             샘플 수       실제 악성 비율
[0.50, 0.60)              2         0.5000
[0.60, 0.80)              3         0.6667
[0.80, 0.90)              2         1.0000
[0.90, 1.00)            100         0.9800

'0.9 이상 확률'로 예측한 100건 중 실제로 악성이 0.9800 — 이 숫자가 0.9에 가까운가(교정)가, 정확도 0.9532와는 별개의 질문입니다.


## 6. 이 숫자로 '반박 가능한 질문' 만들기 (3점 리뷰)

본문 §"손으로 한 번"의 가상 발표(MLP 0.95 > GBDT 0.94 → "MLP가 더 좋다")에,
위 숫자를 대입한 3점 리뷰 질문을 실제로 만들어봅니다 — 8.2절의 3점 기준
(**관찰 + 이 학기 개념(장·절) + 왜 문제인지**)을 그대로 따릅니다.

In [7]:
d_lr, d_mlp, d_gb = (accuracy_score(yte, (proba_te > 0.5)),
                     accuracy_score(yte, (prob_mlp > 0.5)),
                     accuracy_score(yte, (gb.predict_proba(Xte)[:, 1] > 0.5)))
q = (
  f"발표의 'MLP {d_mlp:.4f} > GBDT {d_gb:.4f}'는 1%p 남짓한 정확도 차이인데, "
  f"같은 모델(같은 가중치)의 test 100건 하위샘플만 바꿔도 정확도는 "
  f"{accs.min():.4f}~{accs.max():.4f}로 흔들립니다(이 실험 §4). "
  f"Ch06.3의 선택 편향(하이퍼파라미터 후보 m개 중 val에서 max를 고르는 부풀림, "
  f"m≈30~90이면 ≈0.04~0.05)과 같은 크기의 차이를 'MLP가 더 좋다'는 근거로 쓰실 수 있나요? "
  f"같은 val 분할에서 F1·PR-AUC로 비교한 표를 Ch07.3의 GBDT와 함께 보여주세요 — "
  f"정확도가 아니라 F1/PR-AUC로 순서가 달라지는지도 함께."
)
print(q)


발표의 'MLP 0.9532 > GBDT 0.9415'는 1%p 남짓한 정확도 차이인데, 같은 모델(같은 가중치)의 test 100건 하위샘플만 바꿔도 정확도는 0.9300~0.9900로 흔들립니다(이 실험 §4). Ch06.3의 선택 편향(하이퍼파라미터 후보 m개 중 val에서 max를 고르는 부풀림, m≈30~90이면 ≈0.04~0.05)과 같은 크기의 차이를 'MLP가 더 좋다'는 근거로 쓰실 수 있나요? 같은 val 분할에서 F1·PR-AUC로 비교한 표를 Ch07.3의 GBDT와 함께 보여주세요 — 정확도가 아니라 F1/PR-AUC로 순서가 달라지는지도 함께.


## 요약

- **정확도 차이는 표본 잡음 규모**: 같은 모델·같은 가중치에서 test 100건만
  바꿔도 정확도가 §4의 범위에서 흔들리고, 세 모델 간 차이(§2)는 그보다 작거나
  같은 크기 — "정확도 1위"는 모델이 아니라 **표본의 선택**에 의존할 수 있다.
- **에폭 수는 val 곡선이 정한다**: val 곡선이 포화(§3)된 뒤에도 train만
  계속 학습하는 에폭은 Ch08.1 M3("에폭도 하이퍼파라미터" — val 곡선으로만
  정한다) 기준으로 *val 성능을 지키지 않는* 에폭이다 — "에폭 50"이라는
  숫자에 val 곡선이 뒷받침되어야 3점 리뷰를 통과한다.
- **교정은 정확도와 별개**: "0.99 확률로 예측한 것"의 실제 비율(§5)이
  0.99에 가까워야 확률을 위험도 결정에 쓸 수 있다.
- **반박 가능한 질문** = 관찰(정확도만 비교) + 개념(Ch06.3 선택 편향,
  Ch07.3 GBDT, Ch09.3 early stopping) + "왜 문제인지"(1%p는 잡음 범위)를
  채운 8.2절 3점 기준의 질문.